In [14]:
# import numpy as np

# import os
# from preprocessing.image_utils import extract_features

# def extract_and_save_npy(image_dir, model_cnn, output_dir):
        
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
        
#     images_file = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

#     features = []
#     names = []

#     for img_name in (images_file):
#         img_path = os.path.join(image_dir, img_name)
#         feat = extract_features(model_cnn, img_path)

#         features.append(feat)
#         names.append(img_name)

#     features_matrix = np.array(features)
#     names_array = np.array(names)

#     np.save(os.path.join(output_dir, "features.npy"), features_matrix)
#     np.save(os.path.join(output_dir, "image_names.npy"), names_array)

#     print("Selesai! Fitur dan Nama Gambar telah disimpan.")


In [15]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model

image_dir = "../../dataset/Images"
output_dir = "../../images_feature"

base_model = InceptionV3(weights='imagenet')
model_cnn = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

features_path = os.path.join(output_dir, "features.npy")
names_path = os.path.join(output_dir, "image_names.npy")

# if not os.path.exists(features_path) or not os.path.exists(names_path):
#     print("Memulai proses ekstraksi...")
#     extract_and_save_npy(image_dir, model_cnn, output_dir)
# else:
#     print("Load Feature...")


features_matrix = np.load(features_path)
image_names = np.load(names_path)

image_features_map = dict(zip(image_names, features_matrix))


#### Captions

In [16]:
from utils.text_utils import CaptionPreprocessor

text_util = CaptionPreprocessor()
text_util.build_vocabulary("../../dataset/captions.txt")
map_image_cap = text_util.get_image_to_captions_mapping("../../dataset/captions.txt")
data = {}

for image in map_image_cap:
    captions = map_image_cap[image]
    seq = text_util.texts_to_sequences(captions=captions)
    padded_seq = text_util.pad_sequences(sequences=seq)
    data[image] = padded_seq




### Bagian 3

In [17]:
import tensorflow as tf
import time
import pandas as pd
from utils.train_utils import DataGenerator
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, SimpleRNN, Concatenate, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Masking

def build_model(lstm: bool, layers: int, hidden_state_count: int, vocab_size: int, sequence_length: int = 35):
    image_input = Input(shape=(2048,), name="Image_Input")
    caption_input = Input(shape=(sequence_length,), name="Caption_Input")

    image_projection = Dense(256, activation='relu')(image_input)
    image_projection = Reshape((1, 256))(image_projection)
    image_projection = Masking(mask_value=0.0)(image_projection) 

    caption_embedding = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input)

    merged = Concatenate(axis=1)([image_projection, caption_embedding])

    x = merged
    for i in range(layers):
        if lstm:
            x = LSTM(hidden_state_count, return_sequences=True, name=f"LSTM_Layer_{i+1}")(x)
        else:
            x = SimpleRNN(hidden_state_count, return_sequences=True, name=f"RNN_Layer_{i+1}")(x)

    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    
    outputs = Dense(vocab_size, activation='softmax', name="Output_Layer")(x)

    model = Model(inputs=[image_input, caption_input], outputs=outputs)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.summary()
    return model

In [18]:
variations = [
    {"layers": 1, "hidden": 128, "name": "Shallow_Small"},
    {"layers": 2, "hidden": 128, "name": "Deep_Small"},
    {"layers": 3, "hidden": 128, "name": "VeryDeep_Small"},
    {"layers": 1, "hidden": 256, "name": "Shallow_Mid"},
    {"layers": 1, "hidden": 512, "name": "Shallow_Large"},
    {"layers": 2, "hidden": 512, "name": "Deep_Large"},
]

# Tentukan parameter umum
VOCAB_SIZE = text_util.vocab_size
SEQ_LEN = text_util.sequence_length
EPOCHS = 5
BATCH_SIZE = 64

In [14]:
all_images = list(data.keys())
np.random.seed(42)
np.random.shuffle(all_images)

train_keys = all_images[:6000]
val_keys   = all_images[6000:7000]
test_keys  = all_images[7000:8000]

In [15]:
train_data = {k: data[k] for k in train_keys}
val_data   = {k: data[k] for k in val_keys}
test_data  = {k: data[k] for k in test_keys}

In [16]:
history_logs = []
os.makedirs("weights", exist_ok=True)

for is_lstm in [False, True]:
    arch_name = "LSTM" if is_lstm else "SimpleRNN"
    
    for var in variations:
        model_name = f"{arch_name}_{var['name']}_L{var['layers']}_H{var['hidden']}"
        print(f"\n{'='*50}\nMemulai Training: {model_name}\n{'='*50}")
        
        model = build_model(
            lstm=is_lstm, 
            layers=var['layers'], 
            hidden_state_count=var['hidden'], 
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN
        )
        
        train_gen = DataGenerator(
            mapping_data=train_data,         # <-- ganti dari data ke train_data
            image_features=image_features_map, 
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN,
            batch_size=BATCH_SIZE
        ).generate()

        val_gen = DataGenerator(             # <-- tambahkan val_gen
            mapping_data=val_data,
            image_features=image_features_map,
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN,
            batch_size=BATCH_SIZE
        ).generate()
        
        start_time = time.time()
        history = model.fit(
            train_gen, 
            steps_per_epoch=len(train_data) // BATCH_SIZE,   # <-- sesuaikan
            validation_data=val_gen,                          # <-- tambahkan
            validation_steps=len(val_data) // BATCH_SIZE,    # <-- tambahkan
            epochs=EPOCHS,
            verbose=1
        )
        end_time = time.time()
        
        model.save_weights(f"weights/{model_name}.h5")
        
        duration = end_time - start_time
        history_logs.append({
            "model_type": arch_name,
            "variation_name": var['name'],
            "layers": var['layers'],
            "hidden_state": var['hidden'],
            "final_loss": history.history['loss'][-1],
            "final_val_loss": history.history['val_loss'][-1],   # <-- tambahkan
            "history_loss": history.history['loss'],             # <-- tambahkan untuk grafik
            "history_val_loss": history.history['val_loss'],     # <-- tambahkan untuk grafik
            "training_time_sec": duration
        })

df_results = pd.DataFrame(history_logs)
df_results.to_csv("hasil_variasi_model.csv", index=False)
print("\nEksperimen Selesai! Hasil disimpan di hasil_variasi_model.csv")


Memulai Training: SimpleRNN_Shallow_Small_L1_H128
Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Image_Input (InputLayer)    [(None, 2048)]               0         []                            
                                                                                                  
 dense_4 (Dense)             (None, 256)                  524544    ['Image_Input[0][0]']         
                                                                                                  
 reshape_4 (Reshape)         (None, 1, 256)               0         ['dense_4[0][0]']             
                                                                                                  
 Caption_Input (InputLayer)  [(None, 35)]                 0         []                            
                                         

### Bagian 4

In [20]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
import numpy as np
import os
from utils.text_utils import CaptionPreprocessor
from preprocessing.image_utils import extract_features

# Load Pre-trained CNN Encoder
base_model = InceptionV3(weights='imagenet')
cnn_encoder = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

# Load Pre-trained RNN Decoder
rnn_model = build_model(lstm=False, layers=1, hidden_state_count=128, vocab_size=VOCAB_SIZE, sequence_length=SEQ_LEN)
rnn_model.load_weights("weights/SimpleRNN_Shallow_Small_L1_H128.h5")

# Load Pre-trained LSTM Decoder
lstm_model = build_model(lstm=True, layers=1, hidden_state_count=128, vocab_size=VOCAB_SIZE, sequence_length=SEQ_LEN)
lstm_model.load_weights("weights/LSTM_Shallow_Small_L1_H128.h5")

# Caption Generation Function
def generate_caption(image_path, model_eval, cnn_extractor, text_processor, max_len):
    # Extract image features
    img_feature = extract_features(cnn_extractor, image_path)
    img_feature = np.expand_dims(img_feature, axis=0)  # Adjust dimensions to (1, 2048)

    # Initialize input text with start token
    in_text = '<start>'

    for _ in range(max_len):
        # Convert text to sequence and pad
        sequence = text_processor.texts_to_sequences([in_text])
        padded_sequence = text_processor.pad_sequences(sequence, max_len=max_len)

        # Predict next word probabilities
        predictions = model_eval.predict([img_feature, padded_sequence], verbose=0)
        current_seq_length = len(sequence[0])
        next_word_idx = np.argmax(predictions[0, current_seq_length - 1, :])

        # Convert index to word
        next_word = text_processor.idx_to_word.get(next_word_idx)

        # Stop if end token is predicted
        if next_word is None or next_word == '<end>':
            break

        # Append predicted word to input text
        in_text += ' ' + next_word

    # Clean up the final caption
    final_caption = in_text.replace('<start>', '').strip()
    return final_caption

# Initialize Text Processor
text_processor = CaptionPreprocessor()
text_processor.build_vocabulary("../../dataset/captions.txt")

# Test Caption Generation with RNN
test_image_path = "../../dataset/Images/1000268201_693b08cb0e.jpg"
rnn_caption = generate_caption(test_image_path, rnn_model, cnn_encoder, text_processor, SEQ_LEN)
print("Caption generated by RNN:", rnn_caption)

# Test Caption Generation with LSTM
lstm_caption = generate_caption(test_image_path, lstm_model, cnn_encoder, text_processor, SEQ_LEN)
print("Caption generated by LSTM:", lstm_caption)

ModuleNotFoundError: No module named 'preprocessing'

In [2]:
print("test")

test


In [19]:
import numpy as np

def generate_caption(image_path, model_eval, cnn_extractor, text_processor, max_len):
    # 1. Ekstrak gambar baru menjadi vektor menggunakan CNN
    img_feature = extract_features(cnn_extractor, image_path)
    img_feature = np.expand_dims(img_feature, axis=0) # Sesuaikan dimensi menjadi (1, 2048)
    
    # 2. Inisialisasi input teks dengan token awal pembuka kalimat
    # Asumsi token yang digunakan di text_utils adalah '<start>' dan '<end>'
    in_text = '<start>'
    
    for _ in range(max_len):
        # Ubah teks yang sudah digenerate menjadi urutan angka dan lakukan padding
        sequence = text_processor.texts_to_sequences([in_text])
        padded_sequence = text_processor.pad_sequences(sequence, max_len=max_len)
        
        # 3. Prediksi probabilitas distribusi kata
        predictions = model_eval.predict([img_feature, padded_sequence], verbose=0)
        
        # Ambil probabilitas kata tertinggi pada posisi akhir kalimat saat ini
        current_seq_length = len(sequence[0])
        next_word_idx = np.argmax(predictions[0, current_seq_length - 1, :])
        
        # Kembalikan indeks menjadi string kata nyata
        next_word = text_processor.idx_to_word.get(next_word_idx)
        
        # Hentikan putaran jika indeks tidak terdaftar atau memprediksi token akhir
        if next_word is None or next_word == '<end>':
            break
            
        # Gabungkan kata prediksi ke dalam kalimat untuk putaran selanjutnya
        in_text += ' ' + next_word
        
    # Bersihkan sisa token operasional pada hasil akhir
    final_caption = in_text.replace('<start>', '').strip()
    return final_caption

# ==============================================================================
# CARA PENGGUNAAN (Silakan jalankan sel ini setelah fungsi di atas)
# ==============================================================================

# 1. Bangun ulang arsitektur model yang sama persis dengan yang ingin Anda evaluasi
# Contoh di sini menggunakan variasi LSTM_Shallow_Small_L1_H128
eval_model = build_model(
    lstm=True, 
    layers=1, 
    hidden_state_count=128, 
    vocab_size=VOCAB_SIZE, 
    sequence_length=SEQ_LEN
)

print('test')

# 2. Muat bobot (weights) dari hasil training Bagian 3
eval_model.load_weights("weights/LSTM_Shallow_Small_L1_H128.h5")

# 3. Uji coba fungsi inferensi dengan gambar dari dataset
# Ganti nama file ini dengan gambar yang ada di folder Anda
test_image_path = "../../dataset/Images/1000268201_693b08cb0e.jpg" 

# 4. Hasilkan kalimatnya
caption_hasil = generate_caption(test_image_path, eval_model, model_cnn, text_util, SEQ_LEN)
print("Caption Asli dari Gambar:", caption_hasil)

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Image_Input (InputLayer)    [(None, 2048)]               0         []                            
                                                                                                  
 dense_1 (Dense)             (None, 256)                  524544    ['Image_Input[0][0]']         
                                                                                                  
 reshape_1 (Reshape)         (None, 1, 256)               0         ['dense_1[0][0]']             
                                                                                                  
 Caption_Input (InputLayer)  [(None, 35)]                 0         []                            
                                                                                            

NameError: name 'extract_features' is not defined